# Phase 5 — NB1 v7: Stage 1 Hierarchical Category Detection (Phase B)

**Goal:** Train Stage 1 with Hierarchical Entity→Attribute decomposition.

Architecture:
- DeBERTa → ContextPooler → Entity Head (6 sigmoid) + 3 Attr Heads (FOOD/DRINKS/RESTAURANT, 3 sigmoid each)
- Attribute loss masked to samples where gold entity is active

**Baseline:** Cat-Aware R5 Val F1=0.7243, Test Cat F1=0.6962, Joint F1=0.6235

**Target:** Cat F1 ≥ 0.74 on test set

**Input:** `lcminhc/semeval-absa-restaurant` (raw XML)

**Output:** Upload `/kaggle/working/outputs_p5_nb1/` as Kaggle dataset `p5-nb1-stage1`
- `stage1_hierarchical_best.pt`
- `category_detection.jsonl`, `sentiment_records.jsonl`
- Training logs

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml iterative-stratification

In [ ]:
import os
import sys
import json
import shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# Wire SemEval XML data
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'Dataset semeval-absa-restaurant not found'
print(f'SemEval input: {KAGGLE_INPUT}')

os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training', exist_ok=True)
os.makedirs('SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant', exist_ok=True)

shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_train.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Restaurant Training/ABSA16_Restaurants_Train_SB1_v2.xml')
shutil.copy(f'{KAGGLE_INPUT}/semeval16_restaurants_test.xml',
            'SemEval-Dataset/SemEval 2016 Task 5/Phase B/Gold Annotation/Restaurant/EN_REST_SB1_TEST.xml.gold')
print('XML files in place.')

## 1. Prepare Data

In [ ]:
!python scripts/01_prepare_data.py

In [ ]:
for f in ['category_detection.jsonl', 'sentiment_records.jsonl']:
    path = f'data/processed/{f}'
    if os.path.exists(path):
        with open(path) as fp:
            lines = fp.readlines()
        n = len(lines)
        print(f'{f}: {n} records')
        if n > 0:
            sample = json.loads(lines[0])
            print(f'  keys: {list(sample.keys())}')
    else:
        print(f'MISSING: {f}')

In [ ]:
import torch
import gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Train — Hierarchical Stage 1 (Phase B)

Config: `stage1_hierarchical.yaml`
- 6-entity head + 3 attribute heads (FOOD/DRINKS/RESTAURANT)
- Attribute loss masked to active gold entities
- encoder_lr=1e-5, head_lr=5e-4, 30 epochs, patience=5

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04a_train_stage1.py --config configs/stage1_hierarchical.yaml

In [ ]:
log_path = 'logs/stage1_hierarchical_training.jsonl'
print('=== Hierarchical Training Log ===')
if os.path.exists(log_path):
    print(f'{"Epoch":<8} {"Train Loss":<12} {"Cat F1":<10} {"Cat P":<10} {"Cat R":<10}')
    print('-' * 52)
    with open(log_path) as f:
        for line in f:
            rec = json.loads(line)
            print(f"{rec['epoch']:<8} {rec['train_loss']:<12.4f} {rec['category_f1']:<10.4f} "
                  f"{rec.get('category_precision', 0):<10.4f} {rec.get('category_recall', 0):<10.4f}")
else:
    print('No log found.')

## 3. Results vs Baseline

In [ ]:
def best_epoch(log_path):
    if not os.path.exists(log_path):
        return None
    best = None
    with open(log_path) as f:
        for line in f:
            rec = json.loads(line)
            if best is None or rec['category_f1'] > best['category_f1']:
                best = rec
    return best

cataware_val_f1 = 0.7243  # Cat-Aware R5 best val F1 (ep26)
cataware_test_f1 = 0.6962  # Cat-Aware R5 test Cat F1
hier = best_epoch('logs/stage1_hierarchical_training.jsonl')

print(f'{"Config":<25} {"Best Epoch":<12} {"Cat F1 (val)":<15} {"vs Cat-Aware"}')
print('-' * 65)
print(f'{"Cat-Aware R5 baseline":<25} {"26":<12} {cataware_val_f1:<15.4f} —')
if hier:
    delta = hier['category_f1'] - cataware_val_f1
    print(f'{"Hierarchical":<25} {hier["epoch"]:<12} {hier["category_f1"]:<15.4f} {delta:+.4f}')
    verdict = 'IMPROVEMENT' if hier['category_f1'] > cataware_val_f1 else 'BELOW BASELINE'
    print(f'\nVerdict: {verdict}')
    print('Next: run NB3 v5 for test set evaluation')

## 4. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb1'
os.makedirs(output_dir, exist_ok=True)

# Hierarchical checkpoint
src = 'checkpoints/stage1_hierarchical/best.pt'
if os.path.exists(src):
    shutil.copy(src, f'{output_dir}/stage1_hierarchical_best.pt')
    print(f'stage1_hierarchical_best.pt: {os.path.getsize(src)/1e6:.1f} MB')
else:
    print('WARNING: hierarchical checkpoint not found')

# Data files (needed by NB2 and NB3)
for fname in ['category_detection.jsonl', 'sentiment_records.jsonl']:
    src = f'data/processed/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{fname}')
        print(f'{fname} copied')

# Logs
if os.path.exists('logs'):
    shutil.copytree('logs', f'{output_dir}/logs', dirs_exist_ok=True)
    print('logs/ copied')

print(f'\nOutputs saved to {output_dir}')
print('Upload as Kaggle dataset: p5-nb1-stage1')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_nb1_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_nb1')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_nb1_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')